In [1]:
import os

os.listdir()

['dirty_cafe_sales.csv']

In [2]:
import pandas as pd

df = pd.read_csv("dirty_cafe_sales.csv")

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Null Count": df.isnull().sum(),
    "Unique Values": df.nunique(),
    "Missing/Invalid Count": df.isnull().sum() + df.astype(str).isin(["ERROR", "UNKNOWN"]).sum()
})

print("Dataset Shape:", df.shape)
print("Duplicate Rows:", df.duplicated().sum())
quality_report

Dataset Shape: (10000, 8)
Duplicate Rows: 0


,Data Type,Null Count,Unique Values,Missing/Invalid Count
Transaction ID,object,0,10000,0
Item,object,333,10,969
Quantity,object,138,7,479
Price Per Unit,object,179,8,533
Total Spent,object,173,19,502
Payment Method,object,2579,5,3178
Location,object,3265,4,3961
Transaction Date,object,159,367,460


In [4]:
for column in df.columns:
    print("\n", column)
    print(df[column].value_counts(dropna=False).head(15))


 Transaction ID
Transaction ID
TXN_9226047    1
TXN_8567525    1
TXN_4583012    1
TXN_6796890    1
TXN_9933628    1
TXN_4302199    1
TXN_5548914    1
TXN_3528020    1
TXN_9668108    1
TXN_8076061    1
TXN_7936002    1
TXN_3124078    1
TXN_6120851    1
TXN_5762440    1
TXN_9954652    1
Name: count, dtype: int64

 Item
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

 Quantity
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

 Price Per Unit
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Name: count, dtype: int64

 Total Spent
Total Spent
6.0     979
12.0    939
3.0     930
4.0     923
20.0    746


In [5]:
df = df.replace(["ERROR", "UNKNOWN"], pd.NA)

df.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [6]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[numeric_columns].dtypes

Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

In [7]:
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

df[numeric_columns].isna().sum()

Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64

In [8]:
categorical_columns = ["Item", "Payment Method", "Location"]

for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

df[categorical_columns].isna().sum()

Item              0
Payment Method    0
Location          0
dtype: int64

In [9]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

df["Transaction Date"].isna().sum()

np.int64(460)

In [10]:
df["Transaction Date"] = df["Transaction Date"].fillna(df["Transaction Date"].mode()[0])

df["Transaction Date"].isna().sum()

np.int64(0)

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[column] < lower) | (df[column] > upper)).sum()
    print(column, ":", outliers)

Quantity : 0
Price Per Unit : 0
Total Spent : 259


In [13]:
df["Total Spent"].describe()

count    10000.00000
mean         8.87795
std          5.86059
min          1.00000
25%          4.00000
50%          8.00000
75%         12.00000
max         25.00000
Name: Total Spent, dtype: float64

In [14]:
df.dtypes

Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

In [15]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Values",
        "Invalid ERROR/UNKNOWN Values"
    ],
    "Before Cleaning": [
        10000,
        8,
        0,
        6576,
        0
    ],
    "After Cleaning": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isna().sum().sum(),
        df.astype(str).isin(["ERROR", "UNKNOWN"]).sum().sum()
    ]
})

cleaning_summary

,Metric,Before Cleaning,After Cleaning
0,Rows,10000,10000
1,Columns,8,8
2,Duplicate Rows,0,0
3,Missing Values,6576,0
4,Invalid ERROR/UNKNOWN Values,0,0


In [16]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Values",
        "ERROR/UNKNOWN Values"
    ],
    "Before Cleaning": [
        10000,
        8,
        0,
        6826,
        3256
    ],
    "After Cleaning": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isna().sum().sum(),
        df.astype(str).isin(["ERROR", "UNKNOWN"]).sum().sum()
    ]
})

cleaning_summary

,Metric,Before Cleaning,After Cleaning
0,Rows,10000,10000
1,Columns,8,8
2,Duplicate Rows,0,0
3,Missing Values,6826,0
4,ERROR/UNKNOWN Values,3256,0


In [17]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

In [18]:
print("Shape:", df.shape)
print("Missing Values:", df.isna().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("\nData Types:")
print(df.dtypes)

Shape: (10000, 8)
Missing Values: 0
Duplicate Rows: 0

Data Types:
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object
